<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/4_Logistic_Regression_Business_or_Leisure_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Logistic Regression: Business or Leisure Bookings

An airline would like to know which bookings come from business travelers. The answer is yes or no, which we
code as **1 (business)** or **0 (leisure)**. Predicting a yes/no answer is called **classification**.

Logistic regression is the standard first model for it: it gives each booking a **probability** of being a
business trip, a number between 0 and 1.

Slides: W3.1, logistic regression.

> **Data.** 983 bookings from 2023, read straight from the course GitHub repository. Only two columns are kept:
> `lead_days` (how many days before departure the trip was booked) and `business` (1 = business trip, 0 = leisure).
> Source: Amadeus air bookings sample, licensed to Embry-Riddle.

> **Figures.** The figures in this notebook are pictures made by its full version (`4_Logistic_Regression_Business_or_Leisure_all.ipynb`), so you can focus on the code that fits the model and produces the results.

## 1. Data Prep

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

bookings = pd.read_csv('https://raw.githubusercontent.com/kjmobile/lb2/main/data/booking_lead_days.csv')
bookings.head()

,lead_days,business
0,5,1
1,120,1
2,238,0
3,0,0
4,64,1


In [2]:
print(bookings.shape)
bookings['business'].value_counts()

(983, 2)


business
1    495
0    488
Name: count, dtype: int64

## 2. Look Before You Model

One clue only: how far ahead the trip was booked. Compare the two groups.

In [3]:
# median days booked ahead: 0 = leisure, 1 = business
bookings.groupby('business')['lead_days'].median()

business
0    75.0
1    28.0
Name: lead_days, dtype: float64

Business trips are booked much closer to departure (a median of 28 days) than leisure trips (75 days).
So `lead_days` should help us tell them apart.

**Figure 1**

![Figure 1](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/4_fig1.png)

## 3. Why Not a Straight Line?

We could fit last week's linear regression to the 0/1 column. Look at what it predicts.

In [4]:
from sklearn.linear_model import LinearRegression

line = LinearRegression()
line.fit(bookings[['lead_days']], bookings['business'])

new = pd.DataFrame({'lead_days': [0, 300, 500]})
print(line.predict(new).round(3))

[ 0.653  0.047 -0.357]


For a trip booked 500 days ahead, the straight line predicts **-0.357**. A probability cannot be below 0 (or
above 1), so a straight line is the wrong shape for a yes/no answer.

**Logistic regression** keeps the straight line but passes it through an S-shaped curve, the **logistic
(sigmoid) function**, which always lands between 0 and 1:

$$z = a + b \cdot \text{lead\_days} \qquad\qquad p = \frac{1}{1 + e^{-z}}$$

- $z$ is the straight-line **score**; $p$ is the **probability** of a business trip.
- The **odds** are $p / (1 - p)$: a probability of 0.75 is odds of 3 to 1.
- Underneath the S-curve, the model is still a straight line on the **log-odds** scale: $\log\frac{p}{1-p} = z$.

## 4. Split and Train

As in notebook 0, a prediction model is judged on bookings it has not seen. We keep 30% aside as a **test set**.
`stratify=y` keeps the same share of business trips in both parts.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = bookings[['lead_days']]
y = bookings['business']
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=320, stratify=y)
print('train:', train_X.shape, ' test:', test_X.shape)

train: (688, 1)  test: (295, 1)


In [6]:
logit = LogisticRegression()
logit.fit(train_X, train_y)

b = logit.coef_[0][0]
a = logit.intercept_[0]
print('b (per day):', round(b, 4))
print('a (intercept):', round(a, 3))
print('odds ratio per day:', round(np.exp(b), 3))

b (per day): -0.0095
a (intercept): 0.683
odds ratio per day: 0.991


**Reading b.** $b$ is negative: the further ahead a trip is booked, the lower the chance it is a business trip.
On the odds scale, $e^{b} = 0.991$: each extra day booked ahead multiplies the odds of a business trip by 0.991
(about 1% lower per day).

## 5. Predicted Probabilities

`predict_proba` gives two columns: the probability of class 0 (leisure) and of class 1 (business).
We keep the second.

In [7]:
days = pd.DataFrame({'lead_days': [0, 7, 72, 120, 365]})
days['p_business'] = logit.predict_proba(days)[:, 1].round(3)
days

,lead_days,p_business
0,0,0.664
1,7,0.649
2,72,0.500
3,120,0.388
4,365,0.059


**Figure 2**

![Figure 2](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/4_fig2.png)

To turn a probability into a label, the model uses a **cut-off of 0.5**: above it, "business"; below it, "leisure".
The S-curve crosses 0.5 where the score $z = a + b \cdot \text{lead\_days}$ is 0, that is at $-a / b$ days.

In [8]:
print('P(business) = 0.5 at', round(-a / b), 'days booked ahead')

P(business) = 0.5 at 72 days booked ahead


## 6. How Good Is It?

**Accuracy** is the share of test bookings the model labels correctly. On its own it means little, so compare it
with a **baseline**: always guessing the larger group.

In [9]:
accuracy = logit.score(test_X, test_y)
baseline = max(test_y.mean(), 1 - test_y.mean())
print('test accuracy:', round(accuracy, 3))
print('baseline (always guess the larger group):', round(baseline, 3))

test accuracy: 0.664
baseline (always guess the larger group): 0.505


In [10]:
from sklearn.metrics import confusion_matrix

# rows: what the booking really was (0 leisure, 1 business); columns: what the model said
confusion_matrix(test_y, logit.predict(test_X))

array([[ 72,  74],
       [ 25, 124]], dtype=int64)

The **confusion matrix** counts the right and wrong labels. Of 149 business trips in the test set, the model
catches 124; but it also calls 74 of 146 leisure trips "business". One clue (lead time) helps, but it is not enough
to separate the two groups cleanly.

### Your turn

Many airlines sell their cheapest fares only to trips booked at least 21 days ahead. How likely is a booking made
exactly 21 days ahead to be a business trip?

1. In the cell below, fill in the blank (`___`) and run it.
2. Then finish this sentence:
   *A trip booked 21 days ahead is a business trip with probability ___, so the model labels it ___.*

<details><summary>Show answer</summary>

```python
booking_21 = pd.DataFrame({'lead_days': [21]})
p_21 = logit.predict_proba(booking_21)[0, 1]
print('P(business) at 21 days:', round(p_21, 3))
```

*A trip booked 21 days ahead is a business trip with probability **0.619**, so the model labels it business (above the 0.5 cut-off). The 21-day fare rule sits on the business side of the curve.*
</details>

In [ ]:
booking_21 = pd.DataFrame({'lead_days': [___]})
p_21 = logit.predict_proba(booking_21)[0, 1]
print('P(business) at 21 days:', round(p_21, 3))

## 7. Summary

| Step | What we found |
|---|---|
| The question | Is a booking a business trip (1) or leisure (0)? A classification problem |
| Straight line | Predicts impossible values (below 0 for trips booked far ahead) |
| Logistic regression | An S-curve on top of a straight score; always between 0 and 1 |
| Coefficient | b = -0.0095 per day; odds ratio 0.991 per extra day booked ahead |
| Cut-off 0.5 | Reached at about 72 days booked ahead |
| Test accuracy | 0.664, against 0.505 for always guessing the larger group |

---
# Challenge

1. At how many days booked ahead does the probability of a business trip fall to 0.25? (Hint: try a few values of
   `lead_days` with `predict_proba`, or solve $z = \log(0.25 / 0.75)$ for `lead_days`.)
2. Lead time alone gives 0.664. What other piece of information in a real booking might help tell business from
   leisure trips? Name two, and say why.

In [12]:
# Your answer